In [1]:
from pyscf import gto, scf, dft, lib
import numpy as np
np.set_printoptions(10, linewidth=150)

## Setups

In [2]:
mol = gto.Mole(atom="O; H 1 0.94; H 1 0.94 2 104.5", basis="def2-TZVP").build()

In [3]:
mf = scf.RKS(mol, xc="TPSS0").run()

converged SCF energy = -76.4430874743456


In [4]:
arrays = {
    "mo_coeff": np.asarray(mf.mo_coeff, order="C"),
    "mo_occ": np.asarray(mf.mo_occ, order="C"),
    "rdm1": mf.make_rdm1(),
    "coords": mf.grids.coords,
    "weights": mf.grids.weights,
}

In [5]:
mo_coeff = np.asarray(mf.mo_coeff, order="C")
mo_occ = np.asarray(mf.mo_occ, order="C")
rdm1 = mf.make_rdm1()
coords = mf.grids.coords
weights = mf.grids.weights

In [6]:
grids = mf.grids

In [7]:
ni = dft.numint.NumInt()

In [8]:
np.savez("h2o", **arrays)

## Get rho by dm

In [9]:
ao = ni.eval_ao(mol, grids.coords, deriv=2)
rho_rho = ni.eval_rho(mol, ao[0], rdm1, xctype="lda")
rho_sigma = ni.eval_rho(mol, ao[0:4], rdm1, xctype="gga")
rho_tau = ni.eval_rho(mol, ao[0:4], rdm1, xctype="mgga", with_lapl=False)
rho_lapl = ni.eval_rho(mol, ao[0:10], rdm1, xctype="mgga", with_lapl=True)

In [10]:
rho_rho.shape, lib.fp(rho_rho)

((33704,), np.float64(-438.03033480678425))

In [11]:
rho_sigma.shape, lib.fp(rho_sigma)

((4, 33704), np.float64(25704.144800854465))

In [12]:
rho_tau.shape, lib.fp(rho_tau)

((5, 33704), np.float64(17140.30079159))

In [13]:
rho_lapl_ = rho_lapl[[0, 1, 2, 3, 5, 4]]

In [14]:
rho_lapl_.shape, lib.fp(rho_lapl_[0:5]), lib.fp(rho_lapl_[5])

((6, 33704), np.float64(17140.30079159), np.float64(2470300.1875723638))

## xc_eff

### RHO (LDA)

In [15]:
result = ni.eval_xc_eff("LDA_X", rho_rho, deriv=3)

In [16]:
[r.shape for r in result]

[(33704,), (1, 33704), (1, 1, 33704), (1, 1, 1, 33704)]

In [17]:
fps = np.array([
    lib.fp(result[0] * weights),
    lib.fp(result[1] * weights),
    lib.fp(result[2] * rho_rho * weights),
    lib.fp(result[3] * rho_rho * rho_rho * weights),
])
fps

array([-0.0653646142, -0.0871528189, -0.0290509396,  0.0193672931])

### SIGMA (GGA)

In [18]:
result = ni.eval_xc_eff("GGA_X_PBE", rho_sigma, deriv=3)

In [19]:
[r.shape for r in result]

[(33704,), (4, 33704), (4, 4, 33704), (4, 4, 4, 33704)]

In [20]:
fps = np.array([
    lib.fp(result[0] * weights),
    lib.fp(result[1] * weights),
    lib.fp(result[2] * rho_sigma * weights),
    lib.fp(result[3] * rho_sigma * rho_sigma[:, None, :] * weights),
])
fps

array([-0.1652985961, -0.2296325511, -0.1386410873, -0.523551635 ])

### TAU (MGGA)

In [21]:
result = ni.eval_xc_eff("HYB_MGGA_XC_TPSSH", rho_tau, deriv=3)

In [22]:
[r.shape for r in result]

[(33704,), (5, 33704), (5, 5, 33704), (5, 5, 5, 33704)]

In [23]:
fps = np.array([
    lib.fp(result[0] * weights),
    lib.fp(result[1] * weights),
    lib.fp(result[2] * rho_tau * weights),
    lib.fp(result[3] * rho_tau * rho_tau[:, None, :] * weights),
])
fps

array([-0.1378400498, -0.1893572125, -0.1183454954, -1.0447740367])